# OncoVision: train the lung models on LIDC-IDRI chest CT

Trains three models on the same 641 patients, with the same held-out test patients:
- a **3D nodule finder** for whole CT scans,
- a **2D nodule finder** for single CT images,
- an experimental **benign vs malignant nodule classifier**.

**How to run (important, so nothing gets lost):**
1. Right panel: **Accelerator → GPU T4 x2** (or P100) and **Internet → On**.
2. Top right: **Save Version → Save & Run All (Commit) → Save**. Do *not* use the normal Run All.
3. Close the tab if you like. It runs in the background for about 4-5 hours.
4. Later open the notebook page → **Output** tab → download `oncovision_lung.zip` and send it to Claude together with the printed `SCORES` line.

If a run fails, open the version's **Logs** (or the file `lung_log.txt` in Output) and send Claude the last 30 lines.

Data: LIDC-IDRI (The Cancer Imaging Archive, CC BY 3.0), via the public Hugging Face NIfTI conversion totalol/lungct. No login needed.

In [ ]:
!git clone -q -b vaishnavi/optimistic-noether-034odi https://github.com/Vaishnavi220506/Medical.git /kaggle/working/oncovision
%cd /kaggle/working/oncovision
!pip install -q -r requirements.txt huggingface_hub
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU in Settings!")


## 1. Download, prepare and train the 3D model (about 2-3 hours)
Each CT is downloaded, converted (lung window, 1 mm grid, cropped to the lungs) and deleted again, so the 38 GB of raw data never fill the disk. 100 patients are held out: 40 for validation, 60 for the final test.

In [ ]:
!python -m oncovision.training.train_lung3d --hf --prepared-dir /tmp/lung3d --epochs 60 --samples-per-epoch 800 \
    --patch 96 --batch-size 4 --val-cases 40 --test-cases 60 --val-every 5 --workers 2 --prep-workers 2     --out weights/lung_ct/segmenter3d.pt 2>&1 | tee -a /kaggle/working/lung_log.txt


## 2. 2D model for single CT images (about 30 minutes)
Slices of the same training patients; scored on slices of the held-out test patients.

In [ ]:
!python -m oncovision.training.export_slices --prepared-dir /tmp/lung3d --out /tmp/lung2d --split train --lesion-slices 6 --empty-slices 3
!python -m oncovision.training.export_slices --prepared-dir /tmp/lung3d --out /tmp/lung2d_test --split test --lesion-slices 6 --empty-slices 3
!python -m oncovision.training.train_segmenter --profile lung_ct --format maskfolder --data /tmp/lung2d/images --masks /tmp/lung2d/masks \
    --crop lungs --epochs 30 --batch-size 16 --workers 2 --out weights/lung_ct/segmenter.pt 2>&1 | tee -a /kaggle/working/lung_log.txt
!python -m oncovision.training.evaluate segmenter --profile lung_ct --format maskfolder --data /tmp/lung2d_test/images \
    --masks /tmp/lung2d_test/masks --weights weights/lung_ct/segmenter.pt --out /kaggle/working/lung2d_test


## 3. Benign vs malignant nodule classifier (about 15 minutes, experimental)

In [ ]:
!python -m oncovision.training.train_nodule_cls --prepared-dir /tmp/lung3d --epochs 40 --out weights/lung_ct/nodule_classifier.pt 2>&1 | tee -a /kaggle/working/lung_log.txt


## 4. Scores and package

In [ ]:
import torch, json, glob
scores = {}
for path in sorted(glob.glob("weights/lung_ct/*.pt")):
    meta = torch.load(path, map_location="cpu", weights_only=True)["meta"]
    scores[path.split("/")[-1]] = {k: {kk: vv for kk, vv in v.items() if kk != "by_threshold"} for k, v in meta.get("metrics", {}).items()}
for path in glob.glob("/kaggle/working/*2d_test/metrics.json"):
    scores[path.split("/")[-2]] = {k: v for k, v in json.load(open(path)).items() if not isinstance(v, (list, dict)) or k == "dice"}
print("SCORES (send these to Claude):", json.dumps(scores))
!python scripts/export_weights.py --src weights/lung_ct --dst export_weights/lung_ct
!cd /kaggle/working/oncovision/export_weights && rm -f /kaggle/working/oncovision_lung.zip && zip -qr /kaggle/working/oncovision_lung.zip lung_ct  && ls -la /kaggle/working/oncovision_lung.zip
